# Algoritmos de optimización - Seminario

Nombre y Apellidos:   
* Andrés Gómez
* Iván Cañón


Url: https://github.com/Icanongonz/SEMINARIO/blob/colab/Seminario_Algoritmos.ipynb<br>
Problema: **Sesiones de doblaje**

Descripción del problema: Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las
tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de
grabación independientemente del número de tomas que se graben. No es posible grabar más
de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los
servicios de los actores de doblaje sea el menor posible                                        

(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


Respuesta: 

* La estructura elegida para los datos del problema es una matriz de *t* tomas x *a* actores, en cada elemento (toma, actor) un *1* o *0* según si el actor tiene participación o no. Es una estructura simple que nos permitirá calcular los gastos en base a los actores involucrados en las tomas.

* La estructura para representar la solución es un array de *t* tomas, donde cada elemento representa el día que ser realizará la misma. Es una estructura que nos permitirá hacer diferentes asignaciones de los días sin complejidad, cambiar los días de una toma, intercambiarlos, etc. Calcular la limitación de máximo 6 tomas por días es una operación sencilla de acumular las ocurrencias de cada día.

Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

Respuesta: 

El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. 


<u>Es un problema de minimización</u>

### importación de librerías

In [ ]:
import numpy as np
import pandas as pd

### Carga de datos del problema

In [15]:
# recuperar las tomas/actor desde csv
df = pd.read_csv('Datos problema doblaje (30 tomas, 10 actores).csv', delimiter=';')
df = df.drop('Toma', axis=1)
# generar la matriz toma x actor
problema = df.to_numpy()
print(problema)

[[1 1 1 1 1 0 0 0 0 0]
 [0 0 1 1 1 0 0 0 0 0]
 [0 1 0 0 1 0 1 0 0 0]
 [1 1 0 0 0 0 1 1 0 0]
 [0 1 0 1 0 0 0 1 0 0]
 [1 1 0 1 1 0 0 0 0 0]
 [1 1 0 1 1 0 0 0 0 0]
 [1 1 0 0 0 1 0 0 0 0]
 [1 1 0 1 0 0 0 0 0 0]
 [1 1 0 0 0 1 0 0 1 0]
 [1 1 1 0 1 0 0 1 0 0]
 [1 1 1 1 0 1 0 0 0 0]
 [1 0 0 1 1 0 0 0 0 0]
 [1 0 1 0 0 1 0 0 0 0]
 [1 1 0 0 0 0 1 0 0 0]
 [0 0 0 1 0 0 0 0 0 1]
 [1 0 1 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 0 0 0]
 [1 0 1 0 0 0 0 0 0 0]
 [1 0 1 1 1 0 0 0 0 0]
 [0 0 0 0 0 1 0 1 0 0]
 [1 1 1 1 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 0 0 0]
 [1 1 0 1 0 0 0 0 0 1]
 [1 0 1 0 1 0 0 0 1 0]
 [0 0 0 1 1 0 0 0 0 0]
 [1 0 0 1 0 0 0 0 0 0]
 [1 0 0 0 1 1 0 0 0 0]
 [1 0 0 1 0 0 0 0 0 0]]


Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

In [172]:
# costo para penalizar las soluciones que no cumplen la restricción
COSTO_RESTRICCION = 10000

# Calcula el costo de una solución.
def calcular_costo(problema, solucion):
    '''
    Calcula el costo de una solución.
    Si se exceden las 6 tomas en un mismo día, habrá un costo extra COSTO_RESTRICCION para penalizar la solución
    '''
    costo = 0
    
    # la cantidad de dias maxima es igual a la cantidad de tomas (es en el caso de que se haga una toma por dia)
    tomas = len(solucion)

    # agrupar las tomas por dia
    dias = [[] for i in range(tomas)]
    for t in range(tomas):
        d = solucion[t]
        dias[d].append(t)
        
    # calcular el costo por dia
    for d in range(len(dias)):
        # array donde cada elemento representa la cantidad de tomas en las que ese actor participará ese día.
        actores = [0] * problema.shape[1]
        # recorremos las tomas de ese día 
        for t in dias[d]:
            for a in range(len(actores)):
                actores[a] += problema[t, a]
        # calculamos el costo en base a la cantidad de actores
        for a in range(len(actores)):
            costo += 1 if actores[a] > 0 else 0 # incrementamos 1 el costo si el actor tiene almenos una toma
                
    # RESTRICCION: calcular un costo muy alto si se excede el maximo de 6 tomas
    for d in range(tomas):
        if len(dias[d]) > 6:
            costo += COSTO_RESTRICCION
    
    return costo

solucion = np.array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 29])
calcular_costo(problema, solucion)
    

10012

Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta

### Algoritmo Genético

In [65]:
#Genera una poblacion inicial de soluciones de tamaño N.
def generar_poblacion(problema,N):
    return [np.random.randint(0, problema.shape[0], problema.shape[0]) for _ in range(N)]

In [66]:
#Evalua la población y devuelve el mejor individuo
def Evaluar_Poblacion(poblacion, problem):
    solucion = None
    costo = np.iinfo(np.int32).max
    for s in poblacion:
        c = calcular_costo(problema, s)
        if c < costo:
            solucion = s
            costo = c
    return solucion, costo

In [166]:
# Funcion de cruce. Recibe una poblacion(lista de soluciones) y devuelve la población ampliada con los hijos.
# Todos los individuos de la población son selecionados para el cruce(si la población es par)
def Cruzar(poblacion, problem, mutacion):
    soluciones = []
    soluciones += poblacion
    # los desordenamos para que el cruce sea al azar
    np.random.shuffle(poblacion)
    for p in range(0, len(poblacion) - 1, 2):
        hijo = Descendencia([poblacion[p], poblacion[p+1]], problem, mutacion)
        soluciones.append(hijo)
    return soluciones

In [167]:
# Funcion para generar hijos a partir de 2 padres:
# Se elige el metodo de 1-punto de corte
# Se aplica mutacion
def Descendencia(padres, problem, mutacion):
    # Se elige el metodo de 1-punto de corte
    punto = np.random.randint(0, len(padres[0]))
    hijo = np.concatenate([padres[0][:punto], padres[1][punto:]])
    # se aplica mutacion
    hijo = Mutar(hijo, mutacion)
    return hijo

In [168]:
# Funcion de mutación. Al azar se aplica alguna de estas operaciones:
# - se elije una toma al azar y se le asigna un día al azar
# - se intercambian los dias de dos tomas
# Se hace mutaciones mutacion% de las veces, si no hay mutacion se retorna None
def Mutar(solucion, mutacion):
    # mutacion solo el % de la veces
    if mutacion > np.random.uniform():
        operacion = np.random.randint(0, 2)
        if operacion == 0:
            # se elije una toma al azar y se le asigna un día al azar
            toma = np.random.randint(0, len(solucion))
            solucion[toma] = np.random.randint(0, len(solucion))
        elif operacion == 1:
            # se intercambian los dias de dos tomas
            t1, t2 = np.random.randint(0, len(solucion)), np.random.randint(0, len(solucion))
            solucion[t1], solucion[t2] = solucion[t2], solucion[t1]
    return solucion

In [171]:
poblacion = generar_poblacion(problema, 3)
#print(poblacion)
poblacion = Cruzar(poblacion, problema, 0.1)
#Evaluar_Poblacion(poblacion,problema)

#hijo = Descendencia([poblacion[0], poblacion[1]], problema, 0.1)
#len(hijo)
len(poblacion)

4

(*)Calcula la complejidad del algoritmo

Respuesta

Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

Aplica el algoritmo al juego de datos generado

Respuesta

Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta